# Strategy analysis example

Debugging a strategy can be time-consuming. Freqtrade offers helper functions to visualize raw data.
The following assumes you work with SampleStrategy, data for 5m timeframe from Binance and have downloaded them into the data directory in the default location.
Please follow the [documentation](https://www.freqtrade.io/en/stable/data-download/) for more details.

## Setup

### Change Working directory to repository root

In [15]:
import os
from pathlib import Path
# 添加nest_asyncio支持，解决Jupyter中的事件循环冲突
try:
    import nest_asyncio
    nest_asyncio.apply()
    print("已应用nest_asyncio解决事件循环嵌套问题")
except ImportError:
    print("警告: 未安装nest_asyncio，在Jupyter中可能出现事件循环冲突。请运行 pip install nest_asyncio")


# Change directory
# Modify this cell to insure that the output shows the correct path.
# Define all paths relative to the project root shown in the cell output
project_root = "somedir/freqtrade"
i = 0
try:
    os.chdir(project_root)
    if not Path("LICENSE").is_file():
        i = 0
        while i < 4 and (not Path("LICENSE").is_file()):
            os.chdir(Path(Path.cwd(), "../"))
            i += 1
        project_root = Path.cwd()
except FileNotFoundError:
    print("Please define the project root relative to the current directory")
print(Path.cwd())

已应用nest_asyncio解决事件循环嵌套问题
Please define the project root relative to the current directory
/Users/imm-international/freqtrade


### Configure Freqtrade environment

In [16]:
from freqtrade.configuration import Configuration
from download_data import download_data

# Customize these according to your needs.

# Initialize empty configuration object
config = Configuration.from_files([Path.cwd()/"freqai_config.json"])
# print(config)
# Optionally (recommended), use existing configuration file
# config = Configuration.from_files(["user_data/config.json"])
# download_data(Path.cwd()/"freqai_config.json")

# 下载数据
success = download_data(config)

print('数据下载完成！')
# Define some constants
config["timeframe"] = "15m"
# Name of the strategy class
# config["strategy"] = "SampleStrategy"
# Location of the data
data_location = config["datadir"]
# Pair to analyze - Only use one pair here
pair = "BTC/USDT"

2025-04-14 18:06:09,166 - freqtrade.configuration.load_config - INFO - Using config: /Users/imm-international/freqtrade/freqai_config.json ...

2025-04-14 18:06:09,170 - freqtrade.loggers - INFO - Enabling colorized output.

2025-04-14 18:06:09,171 - root - INFO - Logfile configured

2025-04-14 18:06:09,173 - freqtrade.loggers - INFO - Verbosity set to 0

2025-04-14 18:06:09,175 - freqtrade.configuration.configuration - INFO - Using user-data directory: /Users/imm-international/freqtrade/user_data ...

...

2025-04-14 18:06:09,177 - freqtrade.exchange.check_exchange - INFO - Checking exchange...

2025-04-14 18:06:09,186 - freqtrade.exchange.check_exchange - INFO - Exchange "okx" is officially supported by the Freqtrade development team.

2025-04-14 18:06:09,188 - freqtrade.configuration.configuration - INFO - Using pairlist from configuration.

2025-04-14 18:06:09,189 - freqtrade.configuration.load_config - INFO - Using config: {'exchange': {'name': 'okx', 'key': '', 'secret': '', 'ccxt_config': {}, 'ccxt_async_config': {}, 'pair_whitelist':
['BTC/USDT', 'ETH/USDT'], 'pair_blacklist': []}, 'dataformat_ohlcv': 'feather', 'dataformat_trades': 'feather', 'datadir': PosixPath('/Users/imm-international/freqtrade/user_data/data/okx'), 
'dry_run': True, 'trading_mode': <TradingMode.SPOT: 'spot'>, 'stake_currency': 'USDT', 'stake_amount': 'unlimited', 'max_open_trades': 3, 'fiat_display_currency': 'USD', 'strategy': 'FreqAIStrategy', 
'strategy_path': 'strategies', 'freqaimodel': 'LightGBMClassifier', 'timeframe': '15m', 'timerange': '20220101-20220331', 'new_pairs_days': 30, 'freqai': {'enabled': True, 'purge_old_models': 2, 
'train_period_days': 30, 'backtest_period_days': 7, 'identifier': 'lightgbm_classifier', 'feature_parameters': {'include_timeframes': ['15m', '1h', '4h'], 'include_corr_pairlist': ['BTC/USDT', 
'ETH/USDT'], 'label_period_candles': 24, 'include_shifted_candles': 2, 'DI_threshold': 0.9, 'weight_factor': 0.9, 'principal_component_analysis': False, 'use_SVM_to_remove_outliers': True, 
'indicator_periods_candles': [10, 20, 30], 'plot_feature_importances': True}, 'data_split_parameters': {'test_size': 0.25, 'random_state': 42, 'shuffle': True, 'stratify': True}, 
'model_training_parameters': {'n_estimators': 800, 'learning_rate': 0.01, 'max_depth': 6, 'objective': 'binary', 'verbosity': 1, 'boosting_type': 'gbdt', 'random_state': 42, 'num_leaves': 31}, 
'model_save_parameters': {'max_fit_models': 3}}, 'internals': {'process_throttle_secs': 5}, 'config_files': ['/Users/imm-international/freqtrade/freqai_config.json'], 'pairlists': [], 
'original_config': {'exchange': {'name': 'okx', 'key': '', 'secret': '', 'ccxt_config': {}, 'ccxt_async_config': {}, 'pair_whitelist': ['BTC/USDT', 'ETH/USDT'], 'pair_blacklist': []}, 
'dataformat_ohlcv': 'feather', 'dataformat_trades': 'feather', 'datadir': 'user_data/data', 'dry_run': True, 'trading_mode': 'spot', 'stake_currency': 'USDT', 'stake_amount': 'unlimited', 
'max_open_trades': 3, 'fiat_display_currency': 'USD', 'strategy': 'FreqAIStrategy', 'strategy_path': 'strategies', 'freqaimodel': 'LightGBMClassifier', 'timeframe': '15m', 'timerange': 
'20220101-20220331', 'new_pairs_days': 30, 'freqai': {'enabled': True, 'purge_old_models': 2, 'train_period_days': 30, 'backtest_period_days': 7, 'identifier': 'lightgbm_classifier', 
'feature_parameters': {'include_timeframes': ['15m', '1h', '4h'], 'include_corr_pairlist': ['BTC/USDT', 'ETH/USDT'], 'label_period_candles': 24, 'include_shifted_candles': 2, 'DI_threshold': 0.9, 
'weight_factor': 0.9, 'principal_component_analysis': False, 'use_SVM_to_remove_outliers': True, 'indicator_periods_candles': [10, 20, 30], 'plot_feature_importances': True}, 'data_split_parameters': 
{'test_size': 0.25, 'random_state': 42, 'shuffle': True, 'stratify': True}, 'model_training_parameters': {'n_estimators': 800, 'learning_rate': 0.01, 'max_depth': 6, 'objective': 'binary', 
'verbosity': 1, 'boosting_type': 'gbdt', 'random_state': 42, 'num_leaves': 31}, 'model_save_parameters': {'max_fit_models': 3}}, 'internals': {'process_throttle_secs': 5}, 'config_files': 
['/Users/imm-international/freqtrade/freqai_config.json'], 'pairlists': []}, 'verbosity': 0, 'print_colorized': True, 'runmode': <RunMode.OTHER: 'other'>, 'user_data_dir': 
PosixPath('/Users/imm-international/freqtrade/user_data'), 'exportfilename': PosixPath('/Users/imm-international/freqtrade/user_data/backtest_results'), 'candle_type_def': <CandleType.SPOT: 'spot'>, 
'margin_mode': <MarginMode.NONE: ''>, 'pairs': ['BTC/USDT', 'ETH/USDT']} ...

TypeError: expected str, bytes or os.PathLike object, not dict

In [11]:
# Load data using values set above
from freqtrade.data.history import load_pair_history
from freqtrade.enums import CandleType


candles = load_pair_history(
    datadir=data_location,
    timeframe=config["timeframe"],
    pair=pair,
    data_format="json",  # Make sure to update this to your data
    candle_type=CandleType.SPOT,
)

# Confirm success
print(f"Loaded {len(candles)} rows of data for {pair} from {data_location}")
candles.head()

2025-04-14 18:04:47,189 - freqtrade.data.history.datahandlers.idatahandler - WARNING - No history for BTC/USDT, spot, 15m found. Use `freqtrade download-data` to download the data

Loaded 0 rows of data for BTC/USDT from /Users/imm-international/freqtrade/user_data/data/okx


,date,open,high,low,close,volume


## Load and run strategy
* Rerun each time the strategy file is changed

In [12]:
# Load strategy using values set above
from freqtrade.data.dataprovider import DataProvider
from freqtrade.resolvers import StrategyResolver


strategy = StrategyResolver.load_strategy(config)
strategy.dp = DataProvider(config, None, None)
strategy.ft_bot_start()

# Generate buy/sell signals using strategy
df = strategy.analyze_ticker(candles, {"pair": pair})
df.tail()

2025-04-14 18:04:47,426 - freqtrade.resolvers.iresolver - INFO - Using resolved strategy FreqAIStrategy from '/Users/imm-international/freqtrade/strategies/freqai_strategy.py'...

2025-04-14 18:04:47,429 - freqtrade.strategy.hyper - INFO - Found no parameter file.

2025-04-14 18:04:47,432 - freqtrade.resolvers.strategy_resolver - INFO - Override strategy 'timeframe' with value in config file: 15m.

2025-04-14 18:04:47,435 - freqtrade.resolvers.strategy_resolver - INFO - Override strategy 'stake_currency' with value in config file: USDT.

2025-04-14 18:04:47,438 - freqtrade.resolvers.strategy_resolver - INFO - Override strategy 'stake_amount' with value in config file: unlimited.

...

2025-04-14 18:04:47,590 - freqtrade.data.dataprovider - INFO - Increasing startup_candle_count for freqai on 15m to 2910

2025-04-14 18:04:47,594 - freqtrade.data.dataprovider - INFO - Loading data for BTC/USDT 15m from 2021-12-01 16:30:00 to 2022-03-31 00:00:00

2025-04-14 18:04:47,600 - freqtrade.data.history.datahandlers.idatahandler - WARNING - No history for BTC/USDT, spot, 15m found. Use `freqtrade download-data` to download the data

2025-04-14 18:04:47,604 - freqtrade.data.dataprovider - WARNING - No data found for (BTC/USDT, 15m, ).

KeyError: 'date'

### Display the trade details

* Note that using `data.head()` would also work, however most indicators have some "startup" data at the top of the dataframe.
* Some possible problems
    * Columns with NaN values at the end of the dataframe
    * Columns used in `crossed*()` functions with completely different units
* Comparison with full backtest
    * having 200 buy signals as output for one pair from `analyze_ticker()` does not necessarily mean that 200 trades will be made during backtesting.
    * Assuming you use only one condition such as, `df['rsi'] < 30` as buy condition, this will generate multiple "buy" signals for each pair in sequence (until rsi returns > 29). The bot will only buy on the first of these signals (and also only if a trade-slot ("max_open_trades") is still available), or on one of the middle signals, as soon as a "slot" becomes available.  


In [ ]:
# Report results
print(f"Generated {df['enter_long'].sum()} entry signals")
data = df.set_index("date", drop=False)
data.tail()

## Load existing objects into a Jupyter notebook

The following cells assume that you have already generated data using the cli.  
They will allow you to drill deeper into your results, and perform analysis which otherwise would make the output very difficult to digest due to information overload.

### Load backtest results to pandas dataframe

Analyze a trades dataframe (also used below for plotting)

In [ ]:
from freqtrade.data.btanalysis import load_backtest_data, load_backtest_stats


# if backtest_dir points to a directory, it'll automatically load the last backtest file.
backtest_dir = config["user_data_dir"] / "backtest_results"
# backtest_dir can also point to a specific file
# backtest_dir = (
#   config["user_data_dir"] / "backtest_results/backtest-result-2020-07-01_20-04-22.json"
# )

In [ ]:
# You can get the full backtest statistics by using the following command.
# This contains all information used to generate the backtest result.
stats = load_backtest_stats(backtest_dir)

strategy = "SampleStrategy"
# All statistics are available per strategy, so if `--strategy-list` was used during backtest,
# this will be reflected here as well.
# Example usages:
print(stats["strategy"][strategy]["results_per_pair"])
# Get pairlist used for this backtest
print(stats["strategy"][strategy]["pairlist"])
# Get market change (average change of all pairs from start to end of the backtest period)
print(stats["strategy"][strategy]["market_change"])
# Maximum drawdown ()
print(stats["strategy"][strategy]["max_drawdown_abs"])
# Maximum drawdown start and end
print(stats["strategy"][strategy]["drawdown_start"])
print(stats["strategy"][strategy]["drawdown_end"])


# Get strategy comparison (only relevant if multiple strategies were compared)
print(stats["strategy_comparison"])

In [ ]:
# Load backtested trades as dataframe
trades = load_backtest_data(backtest_dir)

# Show value-counts per pair
trades.groupby("pair")["exit_reason"].value_counts()

## Plotting daily profit / equity line

In [ ]:
# Plotting equity line (starting with 0 on day 1 and adding daily profit for each backtested day)

import pandas as pd
import plotly.express as px

from freqtrade.configuration import Configuration
from freqtrade.data.btanalysis import load_backtest_stats


# strategy = 'SampleStrategy'
# config = Configuration.from_files(["user_data/config.json"])
# backtest_dir = config["user_data_dir"] / "backtest_results"

stats = load_backtest_stats(backtest_dir)
strategy_stats = stats["strategy"][strategy]

df = pd.DataFrame(columns=["dates", "equity"], data=strategy_stats["daily_profit"])
df["equity_daily"] = df["equity"].cumsum()

fig = px.line(df, x="dates", y="equity_daily")
fig.show()

### Load live trading results into a pandas dataframe

In case you did already some trading and want to analyze your performance

In [ ]:
from freqtrade.data.btanalysis import load_trades_from_db


# Fetch trades from database
trades = load_trades_from_db("sqlite:///tradesv3.sqlite")

# Display results
trades.groupby("pair")["exit_reason"].value_counts()

## Analyze the loaded trades for trade parallelism
This can be useful to find the best `max_open_trades` parameter, when used with backtesting in conjunction with a very high `max_open_trades` setting.

`analyze_trade_parallelism()` returns a timeseries dataframe with an "open_trades" column, specifying the number of open trades for each candle.

In [ ]:
from freqtrade.data.btanalysis import analyze_trade_parallelism


# Analyze the above
parallel_trades = analyze_trade_parallelism(trades, "5m")

parallel_trades.plot()

## Plot results

Freqtrade offers interactive plotting capabilities based on plotly.

In [ ]:
from freqtrade.plot.plotting import generate_candlestick_graph


# Limit graph period to keep plotly quick and reactive

# Filter trades to one pair
trades_red = trades.loc[trades["pair"] == pair]

data_red = data["2019-06-01":"2019-06-10"]
# Generate candlestick graph
graph = generate_candlestick_graph(
    pair=pair,
    data=data_red,
    trades=trades_red,
    indicators1=["sma20", "ema50", "ema55"],
    indicators2=["rsi", "macd", "macdsignal", "macdhist"],
)

In [ ]:
# Show graph inline
# graph.show()

# Render graph in a separate window
graph.show(renderer="browser")

## Plot average profit per trade as distribution graph

In [ ]:
import plotly.figure_factory as ff


hist_data = [trades.profit_ratio]
group_labels = ["profit_ratio"]  # name of the dataset

fig = ff.create_distplot(hist_data, group_labels, bin_size=0.01)
fig.show()

Feel free to submit an issue or Pull Request enhancing this document if you would like to share ideas on how to best analyze the data.